# 🧠 Обучение большой модели (~45 млн параметров) на видеокарте

1. Нажми ▶ у ячейки ниже. Разреши доступ к Google Диску — там сохраняется прогресс.
2. Обучение идёт **7 часов** суммарно. Держи вкладку открытой.
3. **Colab отключился?** Ничего не теряется (максимум ~10 минут). Открой эту ссылку снова и нажми ▶ — обучение продолжится с места остановки.

Первый запуск дольше: ~15 минут готовится Википедия (она тоже сохраняется на Диск).

In [ ]:
#@title ▶ Обучать модель (после отключения просто нажми ▶ снова)
import os, torch
SAVE = '/content/drive/MyDrive/ai_training'   # здесь хранится весь прогресс
REPO, BRANCH = '/content/ai', 'ccr-d2dd0252-6ksj1l'
TRAIN_MINUTES = 420   # всего 7 часов обучения (суммарно за все запуски)

if not torch.cuda.is_available():
    print('⚠️ Нет видеокарты! Меню «Среда выполнения» → «Сменить среду выполнения» → T4 GPU → Сохранить. Потом снова ▶')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(SAVE, exist_ok=True)
    if os.path.exists(REPO):
        !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} reset -q --hard FETCH_HEAD
    else:
        !git clone -q -b {BRANCH} https://github.com/r0561952-beep/ai.git {REPO}
    os.chdir(REPO)
    !pip install -q tokenizers
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
    if not os.path.exists(f'{SAVE}/bins/wiki_train.bin'):
        print('Первый запуск: готовлю Википедию (~15 минут)...')
        !python prepare_wiki.py --files 10
    resume = '--resume' if os.path.exists(f'{SAVE}/ckpt/full.pt') else ''
    print('Продолжаю обучение с места остановки' if resume else 'Начинаю обучение с нуля')
    !python train.py {resume} --preset medium --device cuda --mix wiki=0.8,qa=0.2 --batch_size 32 --lr 6e-4 --min_lr 6e-5 --warmup 500 --max_iters 100000000 --max_minutes {TRAIN_MINUTES} --time_schedule --eval_every 1000 --data_dir {SAVE}/bins --ckpt_dir {SAVE}/ckpt --out {SAVE}/model_medium.pt 2>&1 | tee -a {SAVE}/train.log
    if os.path.exists(f'{SAVE}/model_medium.pt'):
        print('\n✅ Обучение завершено! Модель: Google Диск → ai_training → model_medium.pt')

---
### Посмотреть, как идёт обучение
Последние строки журнала (можно запускать в любой момент).

In [ ]:
#@title 📈 Журнал обучения
import os
SAVE = '/content/drive/MyDrive/ai_training'
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
log = f'{SAVE}/train.log'
if os.path.exists(log):
    lines = [l for l in open(log, encoding='utf-8') if l.startswith(('шаг', '  пример', 'продолжаю', 'готово', 'лимит'))]
    print(''.join(lines[-12:]))
else:
    print('Журнала пока нет — сначала запусти обучение.')